# NB09 — Stage B: Final Proposed Model (tuning + epsilon sweep + overfitting proof)

Takes the winning pipeline from **NB08** (`08_best_pipeline.json`) and produces the single **final proposed
model**:

1. **FGM epsilon sweep** (ε ∈ {0.1, 0.3, 0.5, 1.0}) on the winning pipeline — selected by **val** macro-F1.
2. **Final training** of best pipeline + best ε with full per-epoch logging.
3. **Train-vs-validation loss curve** saved to `04_outputs/figures/` — the overfitting proof.
4. **5-seed stability** of the final config (mean ± std, 95% CI).
5. Saves final test predictions + checkpoint for downstream (NB10–15).

Hyperparameter tuning here is legitimate: it is done on **validation only**, on the **single** winning
pipeline, and the final config is reported transparently. Comparative tables (NB06/07/08-fair) stay at a
fixed budget; only this final system is tuned.


In [1]:
import importlib, subprocess, sys
def ensure(p,i=None):
    try: importlib.import_module(i or p)
    except ImportError: subprocess.run([sys.executable,"-m","pip","install","-q",p],check=False)
for pkg,imp in [("transformers","transformers"),("accelerate","accelerate"),("scikit-learn","sklearn"),
                ("pandas","pandas"),("numpy","numpy"),("matplotlib","matplotlib")]: ensure(pkg,imp)
print("deps ok")

deps ok


In [2]:
import os, json, time, random, shutil, warnings, unicodedata, re, gc
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
warnings.filterwarnings("ignore")
HAS_CUDA=torch.cuda.is_available(); HAS_BF16=HAS_CUDA and torch.cuda.is_bf16_supported()
print("cuda:",HAS_CUDA,"| bf16:",HAS_BF16)
if HAS_CUDA: print("GPU:", torch.cuda.get_device_name(0))
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if HAS_CUDA: torch.cuda.manual_seed_all(s)

cuda: True | bf16: True
GPU: NVIDIA A40


In [3]:
def find_repo_root():
    for c in [Path("/workspace/Sarcasm_detection"), Path.cwd()] + list(Path.cwd().resolve().parents):
        if (c/"01_data").exists(): return c.resolve()
    raise RuntimeError("repo root not found.")
ROOT=find_repo_root(); SPLITS=ROOT/"01_data"/"interim"/"splits"
CKPT=ROOT/"03_checkpoints"; TABLES=ROOT/"04_outputs"/"tables"; PRED=ROOT/"04_outputs"/"predictions"
FIGS=ROOT/"04_outputs"/"figures"
for p in [CKPT,TABLES,PRED,FIGS]: p.mkdir(parents=True,exist_ok=True)
assert SPLITS.exists(), "Run NB01 first."

_ZW=dict.fromkeys(map(ord,["\u200b","\u200c","\u200d","\ufeff"]),None)
def norm_key(s):
    if not isinstance(s,str): s="" if pd.isna(s) else str(s)
    s=unicodedata.normalize("NFC",s).translate(_ZW); return re.sub(r"\s+"," ",s).strip().casefold()
def assert_no_leak(tr,va,te,col="text"):
    a={norm_key(x) for x in tr[col]};b={norm_key(x) for x in va[col]};c={norm_key(x) for x in te[col]}
    assert len(a&b)==0 and len(a&c)==0 and len(b&c)==0, "LEAK detected"
def softmax_np(x):
    x=np.asarray(x,np.float64); x=x-x.max(1,keepdims=True); e=np.exp(x); return e/e.sum(1,keepdims=True)
def eval_from_logits(labels,logits,nl):
    labels=np.asarray(labels); preds=np.asarray(logits).argmax(-1)
    m={"accuracy":float(accuracy_score(labels,preds)),
       "macro_f1":float(f1_score(labels,preds,average="macro",zero_division=0)),
       "weighted_f1":float(f1_score(labels,preds,average="weighted",zero_division=0))}
    if nl==2:
        m["binary_f1"]=float(f1_score(labels,preds,average="binary",pos_label=1,zero_division=0))
        m["precision"]=float(precision_score(labels,preds,average="binary",pos_label=1,zero_division=0))
        m["recall"]=float(recall_score(labels,preds,average="binary",pos_label=1,zero_division=0))
    return m,preds
def make_compute_metrics():
    def _cm(ep):
        logits,labels=ep; preds=np.asarray(logits).argmax(-1)
        return {"macro_f1":f1_score(labels,preds,average="macro",zero_division=0)}
    return _cm
def save_predictions(path,texts,labels,logits,nl,extra=None):
    logits=np.asarray(logits); probs=softmax_np(logits); preds=logits.argmax(-1)
    d={"text":[str(t) for t in texts],"gold_label":np.asarray(labels),"pred_label":preds,
       "correct":(np.asarray(labels)==preds).astype(int)}
    for k in range(nl): d[f"logit_{k}"]=logits[:,k]
    for k in range(nl): d[f"prob_{k}"]=probs[:,k]
    df=pd.DataFrame(d)
    if extra:
        for k,v in extra.items(): df[k]=v
    df.to_csv(path,index=False)
class DS(torch.utils.data.Dataset):
    def __init__(self,texts,labels,tok,ml=128):
        self.enc=tok(list(texts),truncation=True,padding="max_length",max_length=ml,return_tensors="pt")
        self.labels=torch.tensor(list(labels),dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self,i):
        it={k:v[i] for k,v in self.enc.items()}; it["labels"]=self.labels[i]; return it
TASK="ben_sarc_binary"; LABEL_COL="label_binary"; NUM_LABELS=2
def load_task():
    def rd(sp):
        df=pd.read_csv(SPLITS/f"{TASK}_{sp}.csv")[["text",LABEL_COL]].dropna(subset=["text"]).copy()
        df["text"]=df["text"].astype(str); df[LABEL_COL]=df[LABEL_COL].astype(int); return df
    tr,va,te=rd("train"),rd("val"),rd("test"); assert_no_leak(tr,va,te); return tr,va,te

In [4]:
# ---- adversarial + lever utilities (identical to NB08) ----
class FGM:
    def __init__(self,model,eps=0.5,emb="word_embeddings"):
        self.m=model; self.eps=eps; self.emb=emb; self.bk={}
    def attack(self):
        for n,p in self.m.named_parameters():
            if p.requires_grad and self.emb in n and p.grad is not None:
                self.bk[n]=p.data.clone(); nm=torch.norm(p.grad)
                if nm!=0 and not torch.isnan(nm): p.data.add_(self.eps*p.grad/nm)
    def restore(self):
        for n,p in self.m.named_parameters():
            if n in self.bk: p.data=self.bk[n]
        self.bk={}
class AWP:
    def __init__(self,model,adv_lr=1.0,adv_eps=1e-3,param="weight"):
        self.m=model; self.lr=adv_lr; self.eps=adv_eps; self.param=param; self.bk={}; self.be={}
    def perturb(self):
        e=1e-6
        for n,p in self.m.named_parameters():
            if p.requires_grad and p.grad is not None and self.param in n:
                self.bk[n]=p.data.clone(); ge=self.eps*p.abs().detach(); self.be[n]=(self.bk[n]-ge,self.bk[n]+ge)
                ng=torch.norm(p.grad); nw=torch.norm(p.data.detach())
                if ng!=0 and not torch.isnan(ng):
                    p.data.add_(self.lr*p.grad*(nw+e)/(ng+e))
                    p.data=torch.min(torch.max(p.data,self.be[n][0]),self.be[n][1])
    def restore(self):
        for n,p in self.m.named_parameters():
            if n in self.bk: p.data=self.bk[n]
        self.bk={}; self.be={}
def get_base(m): return m.base_model
def freeze_lower(model,n,freeze_emb=False):
    base=get_base(model)
    if freeze_emb:
        for p in base.embeddings.parameters(): p.requires_grad=False
    for i in range(min(n,len(base.encoder.layer))):
        for p in base.encoder.layer[i].parameters(): p.requires_grad=False
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
def llrd_groups(model,base_lr,head_lr,decay=0.95,wd=0.01):
    base=get_base(model); L=len(base.encoder.layer); nod=("bias","LayerNorm.weight","layer_norm.weight")
    g=[]; seen=set()
    def add(named,lr):
        dp=[p for n,p in named if p.requires_grad and not any(x in n for x in nod)]
        nd=[p for n,p in named if p.requires_grad and any(x in n for x in nod)]
        if dp: g.append({"params":dp,"lr":lr,"weight_decay":wd})
        if nd: g.append({"params":nd,"lr":lr,"weight_decay":0.0})
        for _,p in named: seen.add(id(p))
    add(list(base.embeddings.named_parameters()), base_lr*(decay**(L+1)))
    for i,layer in enumerate(base.encoder.layer): add(list(layer.named_parameters()), base_lr*(decay**(L-i)))
    add([(n,p) for n,p in model.named_parameters() if id(p) not in seen], head_lr)
    return g
class AdvTrainer(Trainer):
    def __init__(self,*a,adv=None,fgm_eps=0.5,awp_eps=1e-3,llrd=False,llrd_decay=0.95,
                 head_lr=1e-4,base_lr=2e-5,wd=0.01,**k):
        super().__init__(*a,**k); self.adv=adv; self.llrd=llrd; self.llrd_decay=llrd_decay
        self.head_lr=head_lr; self.base_lr=base_lr; self._wd=wd
        self._fgm=FGM(self.model,fgm_eps) if adv in ("fgm","fgm_awp") else None
        self._awp=AWP(self.model,adv_eps=awp_eps) if adv in ("awp","fgm_awp") else None
    def create_optimizer(self):
        if self.optimizer is None and self.llrd:
            self.optimizer=torch.optim.AdamW(llrd_groups(self.model,self.base_lr,self.head_lr,self.llrd_decay,self._wd))
            return self.optimizer
        return super().create_optimizer()
    def training_step(self,model,inputs,num_items_in_batch=None):
        model.train(); inputs=self._prepare_inputs(inputs)
        with self.compute_loss_context_manager(): loss=self.compute_loss(model,inputs)
        if self.args.n_gpu>1: loss=loss.mean()
        self.accelerator.backward(loss)
        if self._fgm is not None:
            self._fgm.attack()
            with self.compute_loss_context_manager(): la=self.compute_loss(model,inputs)
            if self.args.n_gpu>1: la=la.mean()
            self.accelerator.backward(la); self._fgm.restore()
        if self._awp is not None:
            self._awp.perturb()
            with self.compute_loss_context_manager(): lw=self.compute_loss(model,inputs)
            if self.args.n_gpu>1: lw=lw.mean()
            self.accelerator.backward(lw); self._awp.restore()
        return loss.detach()
def build_args(out,epochs,batch,lr,scheduler="linear",warmup=0.1,wd=0.01,ls=0.0,adv=False,grad_clip=1.0):
    bf16=HAS_BF16; fp16=(HAS_CUDA and not HAS_BF16 and not adv)
    common=dict(output_dir=str(out),num_train_epochs=epochs,per_device_train_batch_size=batch,
                per_device_eval_batch_size=batch*2,learning_rate=lr,weight_decay=wd,warmup_ratio=warmup,
                lr_scheduler_type=scheduler,label_smoothing_factor=ls,max_grad_norm=grad_clip,
                save_strategy="epoch",logging_strategy="epoch",save_total_limit=1,
                load_best_model_at_end=True,metric_for_best_model="eval_loss",greater_is_better=False,
                report_to="none",seed=SEED,data_seed=SEED,fp16=fp16,bf16=bf16)
    try: return TrainingArguments(evaluation_strategy="epoch",**common)
    except TypeError: return TrainingArguments(eval_strategy="epoch",**common)

In [5]:
# ---- registry: same names NB08 used, so we can rebuild the winning pipeline ----
BB="csebuetnlp/banglabert"; BB_LARGE="csebuetnlp/banglabert_large"
REGISTRY={
 "P0_fullft":            dict(model=BB, lr=2e-5, max_len=128),
 "P1_fullft_fgm":        dict(model=BB, lr=2e-5, max_len=128, adv="fgm"),
 "P2_fullft_llrd":       dict(model=BB, lr=2e-5, max_len=128, llrd=True),
 "P3_fullft_fgm_llrd":   dict(model=BB, lr=2e-5, max_len=128, adv="fgm", llrd=True),
 "P4_freeze6_fgm":       dict(model=BB, lr=2e-5, max_len=128, adv="fgm", freeze=6),
 "P5_freeze6_fgm_llrd":  dict(model=BB, lr=2e-5, max_len=128, adv="fgm", freeze=6, llrd=True),
 "P6_maxlen192_fgm_llrd":dict(model=BB, lr=2e-5, max_len=192, adv="fgm", llrd=True),
 "P7_fgm_llrd_ls":       dict(model=BB, lr=2e-5, max_len=128, adv="fgm", llrd=True, ls=0.1),
 "P8_large_fgm_llrd":    dict(model=BB_LARGE, lr=1e-5, max_len=128, adv="fgm", llrd=True, batch=16),
}
SEED=42
bp=TABLES/"08_best_pipeline.json"
if bp.exists():
    BEST_NAME=json.load(open(bp))["best_pipeline"]
else:
    BEST_NAME="P3_fullft_fgm_llrd"; print("WARN: 08_best_pipeline.json missing -> default", BEST_NAME)
BASE_CFG=dict(REGISTRY[BEST_NAME]); BASE_CFG["name"]=BEST_NAME
print("FINAL pipeline:", BEST_NAME, "->", BASE_CFG)

# tuning knobs for Stage B
EPS_GRID=[0.1,0.3,0.5,1.0]      # FGM epsilon sweep (only if pipeline uses fgm)
FINAL_EPOCHS=8                  # MATCH NB08 Stage-A budget (P1 at 8 epochs: val 0.8044/test 0.7992)
FINAL_PATIENCE=2   # MATCH NB08 (patience=2)
FINAL_BATCH=BASE_CFG.get("batch",32)   # set 64 if your GPU has the VRAM
tok=AutoTokenizer.from_pretrained(BASE_CFG["model"])

FINAL pipeline: P1_fullft_fgm -> {'model': 'csebuetnlp/banglabert', 'lr': 2e-05, 'max_len': 128, 'adv': 'fgm', 'name': 'P1_fullft_fgm'}


In [6]:
def train_once(cfg, epochs, batch, patience, seed=42, want_history=False):
    set_seed(seed); tr,va,te=load_task()
    ml=cfg.get("max_len",128)
    tr_ds=DS(tr["text"],tr[LABEL_COL],tok,ml); va_ds=DS(va["text"],va[LABEL_COL],tok,ml); te_ds=DS(te["text"],te[LABEL_COL],tok,ml)
    out=CKPT/f"09_{cfg['name']}_seed{seed}"
    if out.exists(): shutil.rmtree(out,ignore_errors=True)
    model=AutoModelForSequenceClassification.from_pretrained(cfg["model"],num_labels=NUM_LABELS)
    if cfg.get("freeze",0): freeze_lower(model,cfg["freeze"],freeze_emb=False)
    args=build_args(out,epochs,batch,cfg["lr"],scheduler=cfg.get("scheduler","linear"),
                    ls=cfg.get("ls",0.0),adv=bool(cfg.get("adv")))
    trn=AdvTrainer(model=model,args=args,train_dataset=tr_ds,eval_dataset=va_ds,
                   compute_metrics=make_compute_metrics(),
                   callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)],
                   adv=cfg.get("adv"),fgm_eps=cfg.get("fgm_eps",0.5),llrd=cfg.get("llrd",False),base_lr=cfg["lr"])
    t0=time.time(); trn.train(); secs=round(time.time()-t0,1)
    vlog=trn.predict(va_ds).predictions; tlog=trn.predict(te_ds).predictions
    vm,_=eval_from_logits(va[LABEL_COL].values,vlog,NUM_LABELS)
    tm,tp=eval_from_logits(te[LABEL_COL].values,tlog,NUM_LABELS)
    hist=None
    if want_history:
        hist=[(h.get("epoch"),h.get("loss"),h.get("eval_loss"),h.get("eval_macro_f1")) for h in trn.state.log_history]
    res=dict(val=vm,test=tm,test_preds=tp,secs=secs,history=hist,
             va=va,te=te,vlog=vlog,tlog=tlog,trainer=trn,model=model,out=out)
    return res

def cleanup(res):
    del res["trainer"], res["model"]; gc.collect()
    if HAS_CUDA: torch.cuda.empty_cache()
    shutil.rmtree(res["out"],ignore_errors=True)

In [7]:
# ---- 1) EPSILON SWEEP (only if the winning pipeline uses FGM) ----
sweep_rows=[]
if BASE_CFG.get("adv")=="fgm":
    for eps in EPS_GRID:
        print(f"\n--- eps={eps} ---")
        cfg=dict(BASE_CFG); cfg["fgm_eps"]=eps; cfg["name"]=f"{BEST_NAME}_eps{eps}"
        try:
            r=train_once(cfg, FINAL_EPOCHS, FINAL_BATCH, FINAL_PATIENCE, seed=42)
            sweep_rows.append({"eps":eps,"val_macro_f1":r["val"]["macro_f1"],"test_macro_f1":r["test"]["macro_f1"],"time_seconds":r["secs"]})
            print(f"  val={r['val']['macro_f1']:.4f} test={r['test']['macro_f1']:.4f}")
            cleanup(r)
        except Exception as e:
            print("  FAILED eps",eps,":",e); torch.cuda.empty_cache() if HAS_CUDA else None
    sweep=pd.DataFrame(sweep_rows).sort_values("val_macro_f1",ascending=False)
    sweep.to_csv(TABLES/"09_epsilon_sweep.csv",index=False)
    BEST_EPS=float(sweep.iloc[0]["eps"]); print("\nBEST eps (by val):", BEST_EPS)
    print(sweep.to_string(index=False,float_format="%.4f"))
else:
    BEST_EPS=None; print("Winning pipeline does not use FGM -> skipping epsilon sweep.")
FINAL_CFG=dict(BASE_CFG)
if BEST_EPS is not None: FINAL_CFG["fgm_eps"]=BEST_EPS
FINAL_CFG["name"]="FINAL_proposed"
json.dump({"pipeline":BEST_NAME,"fgm_eps":BEST_EPS,"epochs":FINAL_EPOCHS,"batch":FINAL_BATCH,
           "config":{k:v for k,v in FINAL_CFG.items()}}, open(TABLES/"09_final_config.json","w"), indent=2)


--- eps=0.1 ---


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1
1,0.552500,0.443627,0.794586
2,0.395200,0.483021,0.787708
3,0.245800,0.543928,0.790469


  val=0.7946 test=0.7935

--- eps=0.3 ---


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Epoch,Training Loss,Validation Loss,Macro F1
1,0.558400,0.449424,0.795023
2,0.405600,0.441024,0.804367
3,0.264100,0.494523,0.788111
4,0.141100,0.567917,0.798031


  val=0.8044 test=0.7992

--- eps=1.0 ---


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1
1,0.569400,0.462343,0.792820
2,0.423100,0.435430,0.800848
3,0.295400,0.460602,0.802382
4,0.173900,0.516407,0.803140


  val=0.8008 test=0.7950

BEST eps (by val): 0.5
   eps  val_macro_f1  test_macro_f1  time_seconds
0.5000        0.8044         0.7992      433.1000
1.0000        0.8008         0.7950      436.6000
0.3000        0.7979         0.7906      324.7000
0.1000        0.7946         0.7935      324.9000


In [8]:
# ---- 2) FINAL TRAIN with full logging + 3) TRAIN-vs-VAL LOSS CURVE (overfitting proof) ----
print("\n=== FINAL MODEL ===", FINAL_CFG)
res=train_once(FINAL_CFG, FINAL_EPOCHS, FINAL_BATCH, FINAL_PATIENCE, seed=42, want_history=True)
fm=res["test"]; print("FINAL TEST:", {k:round(v,4) for k,v in fm.items()})

# save final predictions + model for downstream (NB10-15)
save_predictions(PRED/f"09_FINAL_proposed_{TASK}_val_predictions.csv",
                 res["va"]["text"].values,res["va"][LABEL_COL].values,res["vlog"],NUM_LABELS,extra={"model":"FINAL_proposed","split":"val"})
save_predictions(PRED/f"09_FINAL_proposed_{TASK}_test_predictions.csv",
                 res["te"]["text"].values,res["te"][LABEL_COL].values,res["tlog"],NUM_LABELS,extra={"model":"FINAL_proposed","split":"test"})
json.dump({"test":confusion_matrix(res["te"][LABEL_COL].values,res["test_preds"]).tolist()},
          open(TABLES/"09_FINAL_confusion.json","w"),indent=2)
best_dir=CKPT/"09_FINAL_proposed"/"best_model"; best_dir.mkdir(parents=True,exist_ok=True)
res["trainer"].save_model(str(best_dir)); tok.save_pretrained(str(best_dir))

# loss curve
hist=[h for h in res["history"] if h[0] is not None]
ep_tr=[(e,l) for e,l,_,_ in hist if l is not None]
ep_va=[(e,v) for e,_,v,_ in hist if v is not None]
ep_f1=[(e,f) for e,_,_,f in hist if f is not None]
curve=pd.DataFrame({"epoch":[e for e,_ in ep_tr],"train_loss":[l for _,l in ep_tr]})
vdf=pd.DataFrame({"epoch":[e for e,_ in ep_va],"val_loss":[v for _,v in ep_va]})
fdf=pd.DataFrame({"epoch":[e for e,_ in ep_f1],"val_macro_f1":[f for _,f in ep_f1]})
curve=curve.merge(vdf,on="epoch",how="outer").merge(fdf,on="epoch",how="outer").sort_values("epoch")
curve.to_csv(TABLES/"09_final_loss_curve.csv",index=False)

fig,axes=plt.subplots(1,2,figsize=(11,4.2))
axes[0].plot(curve["epoch"],curve["train_loss"],"o-",label="Train loss")
axes[0].plot(curve["epoch"],curve["val_loss"],"s-",label="Validation loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title("Train vs val loss")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(curve["epoch"],curve["val_macro_f1"],"^-",color="green",label="Val macro-F1")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("macro-F1"); axes[1].set_title("Validation macro-F1")
axes[1].grid(alpha=0.3); axes[1].legend()
fig.suptitle(f"Final proposed model ({BEST_NAME}, eps={FINAL_CFG.get('fgm_eps')}) - epochs<={FINAL_EPOCHS}, patience={FINAL_PATIENCE}")
fig.tight_layout()
fig.savefig(FIGS/"09_final_loss_curve.png",dpi=160); plt.close(fig)
print("saved loss + val-F1 curves -> 04_outputs/figures/09_final_loss_curve.png")

# simple overfitting verdict
try:
    v=curve.dropna(subset=["val_loss"]); 
    min_i=v["val_loss"].idxmin(); last_i=v.index[-1]
    gap=float(v.loc[last_i,"val_loss"]-v.loc[min_i,"val_loss"])
    print(f"val-loss rise after its minimum: {gap:+.4f}  ->",
          "no meaningful overfitting" if gap<0.05 else "some overfitting (early stopping handles it)")
except Exception as e:
    print("verdict skipped:",e)
cleanup(res)


=== FINAL MODEL === {'model': 'csebuetnlp/banglabert', 'lr': 2e-05, 'max_len': 128, 'adv': 'fgm', 'name': 'FINAL_proposed', 'fgm_eps': 0.5}


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1
1,0.558400,0.449424,0.795023
2,0.405600,0.441024,0.804367
3,0.264100,0.494523,0.788111
4,0.141100,0.567917,0.798031


FINAL TEST: {'accuracy': 0.7998, 'macro_f1': 0.7992, 'weighted_f1': 0.7992, 'binary_f1': 0.7879, 'precision': 0.8382, 'recall': 0.7434}
saved loss + val-F1 curves -> 04_outputs/figures/09_final_loss_curve.png
val-loss rise after its minimum: +0.1269  -> some overfitting (early stopping handles it)


In [9]:
# ---- 4) 5-SEED STABILITY of the final config ----
SEEDS=[42,1,7,123,2024]
ms_path=TABLES/"09_final_multiseed.csv"
rows=[]; done=set()
if ms_path.exists():
    prev=pd.read_csv(ms_path); rows=prev.to_dict("records"); done=set(int(s) for s in prev["seed"]); print("resuming seeds:",done)
for seed in SEEDS:
    if seed in done: print("skip seed",seed); continue
    print(f"\n--- final model, seed {seed} ---")
    try:
        r=train_once(FINAL_CFG, FINAL_EPOCHS, FINAL_BATCH, FINAL_PATIENCE, seed=seed)
        rows.append({"seed":seed,"val_macro_f1":r["val"]["macro_f1"],"test_macro_f1":r["test"]["macro_f1"],
                     "test_accuracy":r["test"]["accuracy"],"time_seconds":r["secs"]})
        pd.DataFrame(rows).to_csv(ms_path,index=False); print(f"  test={r['test']['macro_f1']:.4f}")
        cleanup(r)
    except Exception as e:
        print("  FAILED seed",seed,":",e); torch.cuda.empty_cache() if HAS_CUDA else None

ms=pd.read_csv(ms_path); s=ms["test_macro_f1"].values; n=len(s)
mean=float(s.mean()); std=float(s.std(ddof=1)) if n>1 else 0.0; ci=1.96*std/np.sqrt(n) if n>1 else 0.0
summary={"model":"FINAL_proposed","pipeline":BEST_NAME,"fgm_eps":BEST_EPS,"n_seeds":n,
         "mean_macro_f1":round(mean,4),"std":round(std,4),"min":round(float(s.min()),4),
         "max":round(float(s.max()),4),"ci95_low":round(mean-ci,4),"ci95_high":round(mean+ci,4)}
json.dump(summary,open(TABLES/"09_final_multiseed_summary.json","w"),indent=2)
print("\n"+"="*80); print("  FINAL PROPOSED MODEL — 5-seed stability (Ben-Sarc test macro-F1)"); print("="*80)
print(json.dumps(summary,indent=2))
for thr in (0.82,0.85):
    print(f"  mean {'CLEARS' if mean>=thr else 'below'} {thr:.2f}")


--- final model, seed 42 ---


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1
1,0.558400,0.449424,0.795023
2,0.405600,0.441024,0.804367
3,0.264100,0.494523,0.788111
4,0.141100,0.567917,0.798031


  test=0.7992

--- final model, seed 1 ---


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Epoch,Training Loss,Validation Loss,Macro F1
1,0.559100,0.450364,0.793202
2,0.407400,0.448211,0.796808
3,0.261300,0.473716,0.793908
4,0.136800,0.585317,0.797201


  test=0.7933

--- final model, seed 2024 ---


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1
1,0.554400,0.449300,0.794115
2,0.404300,0.448340,0.797062
3,0.262800,0.477509,0.798923
4,0.138400,0.585916,0.798512


  test=0.7946

  FINAL PROPOSED MODEL — 5-seed stability (Ben-Sarc test macro-F1)
{
  "model": "FINAL_proposed",
  "pipeline": "P1_fullft_fgm",
  "fgm_eps": 0.5,
  "n_seeds": 5,
  "mean_macro_f1": 0.796,
  "std": 0.0038,
  "min": 0.7922,
  "max": 0.8009,
  "ci95_low": 0.7927,
  "ci95_high": 0.7994
}
  mean below 0.82
  mean below 0.85
